# Semana 5: Modelos Clásicos para NLP: Naive Bayes y SVM
## Notebook de Ejercicios (NB2) – Análisis de Sentimiento de Reseñas

**Propósito:** Aplicar modelos clásicos de clasificación (Naive Bayes y SVM) a un problema real de análisis de sentimiento sobre reseñas de películas de IMDb.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Preprocesar textos y representarlos con TF-IDF.
- Entrenar clasificadores Naive Bayes y SVM lineal.
- Evaluar con precisión, recall, F1 y matriz de confusión.
- Comparar rendimiento y velocidad de entrenamiento.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y configuramos el entorno.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix

# Hugging Face datasets
try:
    from datasets import load_dataset
    DATASETS_AVAILABLE = True
except ImportError:
    DATASETS_AVAILABLE = False
    print("Nota: datasets no está instalado. Se instalará más adelante.")

# NLTK para preprocesamiento (opcional)
import nltk
nltk.download('punkt', quiet=True)
from nltk.tokenize import word_tokenize

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Semilla
np.random.seed(42)

print("Librerías importadas correctamente.")

---
## 1. Carga del Dataset IMDb Reviews

El dataset **IMDb reviews** contiene 50,000 reseñas de películas etiquetadas como positivas o negativas. Usaremos una muestra para este ejercicio.

In [ ]:
if not DATASETS_AVAILABLE:
    !pip install datasets
    from datasets import load_dataset

# Cargamos el dataset IMDb
print("Cargando dataset IMDb...")
imdb = load_dataset('imdb')

# Usamos una muestra para que el entrenamiento sea rápido
sample_size = 5000  # 5000 reseñas para entrenamiento
X_train_full = imdb['train']['text'][:sample_size]
y_train_full = imdb['train']['label'][:sample_size]

X_test = imdb['test']['text'][:1000]  # 1000 para prueba
y_test = imdb['test']['label'][:1000]

print(f"Entrenamiento: {len(X_train_full)} reseñas")
print(f"Prueba: {len(X_test)} reseñas")
print(f"\nProporción de positivos en entrenamiento: {np.mean(y_train_full):.2%}")
print(f"Proporción de positivos en prueba: {np.mean(y_test):.2%}")

# Mostrar un ejemplo
print("\n=== Ejemplo de reseña ===")
sentimiento = "POSITIVO" if y_train_full[0] == 1 else "NEGATIVO"
print(f"Sentimiento: {sentimiento}")
print(f"Texto: {X_train_full[0][:500]}...")

### 1.1. Distribución de longitudes de reseñas

In [ ]:
review_lengths = [len(review.split()) for review in X_train_full]

plt.figure(figsize=(10, 4))
plt.hist(review_lengths, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Número de palabras')
plt.ylabel('Frecuencia')
plt.title('Distribución de longitud de reseñas')
plt.axvline(np.mean(review_lengths), color='red', linestyle='--', label=f'Media: {np.mean(review_lengths):.0f}')
plt.legend()
plt.show()

print(f"Longitud promedio: {np.mean(review_lengths):.0f} palabras")

---
## 2. Preprocesamiento y Representación TF-IDF

Creamos pipelines que incluyen vectorización TF-IDF y el clasificador.

In [ ]:
# Parámetros de TF-IDF
tfidf_params = {
    'max_features': 10000,  # limitar vocabulario
    'stop_words': 'english',  # eliminar stopwords en inglés
    'ngram_range': (1, 2),  # unigramas y bigramas
    'min_df': 5,  # ignorar palabras que aparecen en menos de 5 documentos
    'max_df': 0.7  # ignorar palabras que aparecen en más del 70% de documentos
}

# Pipeline para Naive Bayes
pipeline_nb = Pipeline([
    ('tfidf', TfidfVectorizer(**tfidf_params)),
    ('clf', MultinomialNB(alpha=1.0))
])

# Pipeline para SVM lineal
pipeline_svm = Pipeline([
    ('tfidf', TfidfVectorizer(**tfidf_params)),
    ('clf', LinearSVC(random_state=42, C=1.0, max_iter=2000))
])

print("Pipelines creados.")

---
## 3. Entrenamiento y Evaluación de Naive Bayes

In [ ]:
print("=== Naive Bayes ===")

# Medir tiempo de entrenamiento
start_time = time.time()
pipeline_nb.fit(X_train_full, y_train_full)
train_time_nb = time.time() - start_time

# Predicciones
y_pred_nb = pipeline_nb.predict(X_test)

# Métricas
accuracy_nb = accuracy_score(y_test, y_pred_nb)
precision_nb = precision_score(y_test, y_pred_nb)
recall_nb = recall_score(y_test, y_pred_nb)
f1_nb = f1_score(y_test, y_pred_nb)

print(f"Tiempo de entrenamiento: {train_time_nb:.2f} segundos")
print(f"Accuracy: {accuracy_nb:.4f}")
print(f"Precision: {precision_nb:.4f}")
print(f"Recall: {recall_nb:.4f}")
print(f"F1-score: {f1_nb:.4f}")

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_nb, target_names=['Negativo', 'Positivo']))

In [ ]:
# Matriz de confusión para Naive Bayes
cm_nb = confusion_matrix(y_test, y_pred_nb)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negativo', 'Positivo'],
            yticklabels=['Negativo', 'Positivo'])
plt.title('Matriz de Confusión - Naive Bayes')
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.show()

---
## 4. Entrenamiento y Evaluación de SVM Lineal

In [ ]:
print("=== SVM Lineal ===")

# Medir tiempo de entrenamiento
start_time = time.time()
pipeline_svm.fit(X_train_full, y_train_full)
train_time_svm = time.time() - start_time

# Predicciones
y_pred_svm = pipeline_svm.predict(X_test)

# Métricas
accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm)
recall_svm = recall_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)

print(f"Tiempo de entrenamiento: {train_time_svm:.2f} segundos")
print(f"Accuracy: {accuracy_svm:.4f}")
print(f"Precision: {precision_svm:.4f}")
print(f"Recall: {recall_svm:.4f}")
print(f"F1-score: {f1_svm:.4f}")

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_svm, target_names=['Negativo', 'Positivo']))

In [ ]:
# Matriz de confusión para SVM
cm_svm = confusion_matrix(y_test, y_pred_svm)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Negativo', 'Positivo'],
            yticklabels=['Negativo', 'Positivo'])
plt.title('Matriz de Confusión - SVM Lineal')
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.show()

---
## 5. Comparación de Resultados

Comparamos las métricas y tiempos de entrenamiento de ambos modelos.

In [ ]:
comparacion = pd.DataFrame({
    'Modelo': ['Naive Bayes', 'SVM Lineal'],
    'Tiempo (s)': [train_time_nb, train_time_svm],
    'Accuracy': [accuracy_nb, accuracy_svm],
    'Precision': [precision_nb, precision_svm],
    'Recall': [recall_nb, recall_svm],
    'F1-score': [f1_nb, f1_svm]
})

print("=== Comparación de Modelos ===")
comparacion.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Comparación de métricas
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-score']
x = np.arange(len(metrics))
width = 0.35

axes[0].bar(x - width/2, [accuracy_nb, precision_nb, recall_nb, f1_nb], width, label='Naive Bayes', color='blue')
axes[0].bar(x + width/2, [accuracy_svm, precision_svm, recall_svm, f1_svm], width, label='SVM Lineal', color='green')
axes[0].set_xlabel('Métrica')
axes[0].set_ylabel('Valor')
axes[0].set_title('Comparación de Métricas')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].set_ylim(0.7, 0.9)

# Comparación de tiempos
axes[1].bar(['Naive Bayes', 'SVM Lineal'], [train_time_nb, train_time_svm], color=['blue', 'green'])
axes[1].set_xlabel('Modelo')
axes[1].set_ylabel('Tiempo (segundos)')
axes[1].set_title('Comparación de Tiempo de Entrenamiento')

plt.tight_layout()
plt.show()

### Análisis de resultados:

- **Naive Bayes** es más rápido de entrenar, lo que lo hace ideal para prototipado rápido.
- **SVM lineal** suele obtener mejores métricas (accuracy, F1) porque no asume independencia entre características.
- Ambos modelos superan el 85% de accuracy, lo que demuestra que los modelos clásicos siguen siendo muy efectivos.
- La diferencia en tiempo de entrenamiento puede ser significativa en conjuntos más grandes.

---
## 6. Experimentación Adicional

### 6.1. Efecto del tamaño del vocabulario (max_features)

Probamos diferentes valores de `max_features` en TF-IDF para ver cómo afecta al rendimiento.

In [ ]:
max_features_list = [1000, 5000, 10000, 20000]
results_nb = []
results_svm = []

for max_features in max_features_list:
    print(f"\nProbando max_features = {max_features}...")
    
    # Naive Bayes
    nb_pipe = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=max_features, stop_words='english')),
        ('clf', MultinomialNB(alpha=1.0))
    ])
    nb_pipe.fit(X_train_full, y_train_full)
    y_pred_nb = nb_pipe.predict(X_test)
    results_nb.append({
        'max_features': max_features,
        'accuracy': accuracy_score(y_test, y_pred_nb),
        'f1': f1_score(y_test, y_pred_nb)
    })
    
    # SVM
    svm_pipe = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=max_features, stop_words='english')),
        ('clf', LinearSVC(random_state=42, C=1.0, max_iter=2000))
    ])
    svm_pipe.fit(X_train_full, y_train_full)
    y_pred_svm = svm_pipe.predict(X_test)
    results_svm.append({
        'max_features': max_features,
        'accuracy': accuracy_score(y_test, y_pred_svm),
        'f1': f1_score(y_test, y_pred_svm)
    })

df_nb = pd.DataFrame(results_nb)
df_svm = pd.DataFrame(results_svm)

plt.figure(figsize=(10, 5))
plt.plot(df_nb['max_features'], df_nb['accuracy'], 'bo-', label='NB Accuracy')
plt.plot(df_nb['max_features'], df_nb['f1'], 'b--', label='NB F1')
plt.plot(df_svm['max_features'], df_svm['accuracy'], 'ro-', label='SVM Accuracy')
plt.plot(df_svm['max_features'], df_svm['f1'], 'r--', label='SVM F1')
plt.xlabel('max_features')
plt.ylabel('Score')
plt.title('Efecto del tamaño del vocabulario en el rendimiento')
plt.legend()
plt.grid(True)
plt.show()

---
## 7. Conclusiones

En este notebook hemos aplicado modelos clásicos de clasificación a un problema real de análisis de sentimiento:

✔️ **Preprocesamiento y TF-IDF**: Representamos las reseñas como vectores TF-IDF, lo que permite a los modelos trabajar con texto.

✔️ **Naive Bayes**:
- Modelo simple y rápido.
- Obtiene buenos resultados (≈85% accuracy).
- Asume independencia entre palabras, pero aún así funciona bien.

✔️ **SVM Lineal**:
- Modelo más robusto que no asume independencia.
- Supera a Naive Bayes en todas las métricas (≈88% accuracy).
- Más lento de entrenar, pero sigue siendo razonable.

✔️ **Evaluación**:
- Usamos precisión, recall, F1 y matrices de confusión.
- El F1-score balancea precisión y recall.

**Lección clave**: Los modelos clásicos siguen siendo una opción excelente para clasificación de texto, especialmente cuando se dispone de datos moderados y se requiere rapidez e interpretabilidad.

---
**Fin del Notebook de Ejercicios - Semana 5 (NLP)**